In [1]:
import os
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys

sys.path.insert(0, "../..")

In [2]:
import chex
import dill
import jax
import jax.numpy as jnp
import json
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import scipy
import torch

from flax import nnx
from torch.utils.data import DataLoader
from typing import Any, NamedTuple
from typing_extensions import Protocol, runtime_checkable

from src.dataset import get_iter
from src.datasets.sum import Addition
from src.decoding import make_autoregressive
from src.utils import parse_dict
from src.verifier import make_compute_returns

# Penzai
from penzai import pz

import IPython

pz.ts.register_as_default()

# Optional automatic array visualization extras:
pz.ts.register_autovisualize_magic()
pz.enable_interactive_context()
pz.ts.active_autovisualizer.set_interactive(pz.ts.ArrayAutovisualizer())


In [3]:
base_path = "/home/bryanpu1/projects/parallel_vs_serial/scaling_jax/results"
algo_name = "addition-max_int_64-no_eos"
run_name = "metastable_ppo-scratch_1M-0_cot_tokens-8x8-11-18-25_09_03_56-deacc19b-7394-432c-aca2-8d5d3592c247"

learner_path = os.path.join(base_path, algo_name, run_name)

In [4]:
eval_seed = 42
num_evals = 1
max_decode_len = 100
num_bits = 10
max_int = 2 ** num_bits
max_batch_size = 1
save_filename = f"res-{run_name}-decode_len_{max_decode_len}-num_evals_{num_evals}.dill"
checkpoint_i = 20

config_dict = json.load(open(os.path.join(learner_path, "config.json"), "r"))
half_precision = config_dict["half_precision"]
dtype = jnp.bfloat16 if half_precision else jnp.float32

# Get dataset
train_max_int = config_dict["dataset_kwargs"]["max_int"]
config_dict["dataset_kwargs"]["max_int"] = 4096
config_dict["gamma"] = 0.99
config = parse_dict(config_dict)
dataset_kwargs = config.dataset_kwargs

num_checkpoints = len(os.listdir(os.path.join(learner_path, "models")))

if config.dataset_name == "curriculum":
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.datasets[-1]["dataset_kwargs"]["max_int"])))
    train_val_ratio = dataset_kwargs.datasets[-1]["dataset_kwargs"]["train_val_ratio"]
    predict_eos = dataset_kwargs.datasets[-1]["dataset_kwargs"]["predict_eos"]
    num_cot_tokens = dataset_kwargs.datasets[-1]["dataset_kwargs"]["num_cot_tokens"]
    right_to_left = dataset_kwargs.datasets[-1]["dataset_kwargs"]["right_to_left"]
    correctness_aware = dataset_kwargs.datasets[-1]["dataset_kwargs"]["correctness_aware"]
    carry_registers = dataset_kwargs.datasets[-1]["dataset_kwargs"]["carry_registers"]
else:
    max_num_bits = int(np.ceil(np.log2(dataset_kwargs.max_int)))
    train_val_ratio = dataset_kwargs.train_val_ratio
    predict_eos = dataset_kwargs.predict_eos
    num_cot_tokens = dataset_kwargs.num_cot_tokens
    right_to_left = dataset_kwargs.right_to_left
    correctness_aware = dataset_kwargs.correctness_aware
    carry_registers = getattr(dataset_kwargs, "carry_registers", False)

dataset = Addition(
    context_len=max_decode_len,
    max_int=max_int,
    train=False,
    seed=eval_seed,
    sequence_type="question_only",
    train_val_ratio=train_val_ratio,
    right_to_left=right_to_left,
    num_repeats=None,
    shuffle=False,
    exact=True,
    predict_eos=predict_eos,
    num_cot_tokens=num_cot_tokens,
    p_curriculum=0.0,
    p_inject_noop=0.0,
    max_noops=0,
    noop_as_pad=False,
    reverse_curriculum=False,
    correctness_aware=correctness_aware,
    carry_registers=carry_registers,
)

EOS TOKEN: 6
TOKEN MAP: {0: 0, 1: 1, 2: 2, 3: 3, 6: 6, 4: 4, 5: 5}


In [5]:
num_pairs = max_int * max_int
num_val = num_pairs - int(np.floor(num_pairs * train_val_ratio))
batch_size = min(num_val, max_batch_size)

out_dim = int(dataset.output_space.n)
data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=0,
)
data_iter = get_iter(data_loader, None, dtype)
batch = next(data_iter)
batch = {
    "sequence": np.repeat(batch["sequence"], num_evals, axis=0),
    "mask": np.repeat(batch["mask"], num_evals, axis=0),
    "target": np.repeat(batch["target"], num_evals, axis=0),
    "pointer_correct": np.repeat(batch["pointer_correct"], num_evals, axis=0),
}

In [6]:
last_step = sorted(os.listdir(os.path.join(learner_path, "models")))[checkpoint_i]
train_state = dill.load(
    open(os.path.join(learner_path, "models", last_step), "rb")
)

model = nnx.merge(
    train_state.graphdef,
    train_state.params,
    train_state.rest,
)
model.set_attributes(deterministic=False, decode=False)

In [7]:
res = model(batch)

In [8]:
intermediates = nnx.pop(model, nnx.Intermediate)

In [9]:
input_length = np.where(batch["sequence"][0] == dataset.eos_token_id)[0][0]

In [10]:
batch

{'sequence': array([[1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 2, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 3,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]]),
 'mask': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1.]], dtype=float32),
 'target': array([[3, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]]),
 'pointer_correct': array([1])}

In [11]:
attn_weights = []
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0][0, :, :input_length, :input_length])

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "head", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"]
)

<Arrayviz rendering>

In [12]:
attn_weights = []
input_length = np.where(batch["sequence"][0] == dataset.eos_token_id)[0][0]
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(np.mean(
        intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0][0, :, :input_length, :input_length],
        axis=0,
    ))

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"]
)

<Arrayviz rendering>

In [13]:
batch["sequence"], batch["target"]

(array([[1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 2, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 3,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]]),
 array([[3, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,
         6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]]))

In [14]:
new_seq = batch["sequence"][:]
new_seq[:, input_length - 1:] = batch["target"][:, :-input_length + 1]

In [15]:
seq_len = np.where(new_seq[0] == dataset.eos_token_id)[0][0]

In [16]:
res = model({
    "sequence": new_seq
})

In [17]:
res[:, input_length - 1:seq_len]

Array([[[-35.272095, -10.331284, -36.525997, -36.530663],
        [-17.768375, -33.27291 , -38.119236, -39.202354],
        [-12.423359, -35.68302 , -37.84762 , -38.19363 ],
        [-10.734095, -36.408047, -37.4023  , -37.53836 ],
        [-30.187614, -20.502981, -37.503548, -38.949978],
        [-13.381362, -35.097847, -38.31208 , -38.502968],
        [-14.367687, -34.448605, -38.44555 , -39.106064],
        [-35.244385, -10.117336, -36.484394, -36.727158],
        [-35.298714,  -9.469633, -36.358845, -36.766754],
        [-35.128517,  -9.395475, -36.481937, -37.038597],
        [-34.827488,  -9.633849, -36.987026, -37.05271 ]]], dtype=float32)

In [18]:
probs = jax.nn.softmax(res[:, input_length - 1:seq_len - 1])
probs

Array([[[1.4734769e-11, 1.0000000e+00, 4.2051397e-12, 4.1855673e-12],
        [9.9999976e-01, 1.8469946e-07, 1.4512206e-09, 4.9129362e-10],
        [1.0000000e+00, 7.9151082e-11, 9.0862465e-12, 6.4285552e-12],
        [1.0000000e+00, 7.0785496e-12, 2.6190560e-12, 2.2858894e-12],
        [6.2228624e-05, 9.9993777e-01, 4.1373355e-08, 9.7396677e-09],
        [1.0000000e+00, 3.7038281e-10, 1.4884252e-11, 1.2297748e-11],
        [1.0000000e+00, 1.9009401e-09, 3.4923477e-11, 1.8040956e-11],
        [1.2230986e-11, 1.0000000e+00, 3.5394214e-12, 2.7765240e-12],
        [6.0613983e-12, 1.0000000e+00, 2.0997314e-12, 1.3964036e-12],
        [6.6723987e-12, 1.0000000e+00, 1.7238481e-12, 9.8797229e-13]]],      dtype=float32)

In [19]:
batch["target"][0][1:probs.shape[1] + 1]

array([1, 0, 0, 0, 1, 0, 1, 1, 1, 1])

In [20]:
intermediates = nnx.pop(model, nnx.Intermediate)

In [21]:
attn_weights = []
for layer_i in intermediates["gpt"]["gpt"]["layers"]:
    attn_weights.append(intermediates["gpt"]["gpt"]["layers"][layer_i]["attention"]["attention_weights"].value[0][0, :, :seq_len, :seq_len])

attn_weights = pz.nx.wrap(
    attn_weights
).tag("layer", "head", "q", "k")
pz.ts.render_array(
    attn_weights,
    rows=["q"],
    columns=["k"]
)

<Arrayviz rendering>